# Module 01: Data Parallelism (DDP)

Read `README.md` for this module before starting.

**Strategy**: One full model copy per GPU. Different data per GPU. Gradients averaged via AllReduce.

## Setup

In [ ]:
!pip install -q accelerate==0.34.0

import torch
import time

assert torch.cuda.device_count() == 2, "Need 2 GPUs. Check Kaggle settings."
print(f"Ready. {torch.cuda.device_count()} GPUs available.")

## The TrainingProfiler (paste this into every module)

In [ ]:
import time
import torch

class TrainingProfiler:
    def __init__(self, batch_size: int, rank: int = 0, log_every: int = 10):
        self.batch_size = batch_size
        self.rank = rank
        self.log_every = log_every
        self._step_start = time.perf_counter()
        self._session_start = time.perf_counter()
        self._losses = []
        self._throughputs = []

    def step(self, step: int, loss: float):
        if self.rank != 0:
            return
        elapsed = time.perf_counter() - self._step_start
        throughput = self.batch_size / elapsed
        self._losses.append(loss)
        self._throughputs.append(throughput)
        if step % self.log_every == 0:
            gpu_info = " | ".join(
                f"GPU{i}={torch.cuda.memory_allocated(i)/1e9:.1f}GB"
                for i in range(torch.cuda.device_count())
            )
            print(f"[Step {step:4d}] loss={loss:.4f} | {throughput:.0f} samples/s | {gpu_info}")
        self._step_start = time.perf_counter()

    def summary(self):
        if self.rank != 0 or not self._losses:
            return
        total_time = time.perf_counter() - self._session_start
        avg_tp = sum(self._throughputs) / len(self._throughputs)
        print(f"\nSummary: {total_time:.1f}s total | {avg_tp:.0f} avg samples/s | final loss={self._losses[-1]:.4f}")
        return avg_tp

## Part 1: Write the DDP Training Script

In Kaggle, multi-process scripts must be written to a file and launched via `torchrun`.
The `%%writefile` magic does this for us.

In [ ]:
%%writefile /kaggle/working/ddp_train.py
"""
DDP Training Script

This file is launched by torchrun which spawns one process per GPU.
Each process runs this entire file independently.
"""
import os
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

# ─── Hyperparameters ──────────────────────────────────────────────────────────
BATCH_SIZE = 64       # Per-GPU batch size
SEQ_LEN = 128
VOCAB_SIZE = 10000
D_MODEL = 256
NUM_EPOCHS = 3
DATASET_SIZE = 1000
LR = 1e-4

# ─── Model ────────────────────────────────────────────────────────────────────
class TransformerClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
        encoder_layer = nn.TransformerEncoderLayer(
            D_MODEL, nhead=8, dim_feedforward=D_MODEL*4,
            dropout=0.1, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=6)
        self.classifier = nn.Linear(D_MODEL, 2)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.classifier(x[:, 0, :])


# ─── Main training function ───────────────────────────────────────────────────
def main():
    # Step 1: Read rank/world_size from environment (set by torchrun)
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    # Step 2: Initialize the distributed process group
    # backend='nccl' is always correct for GPU training
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)

    if rank == 0:
        print(f"Starting DDP training with {world_size} GPUs")
        print(f"Global batch size: {BATCH_SIZE * world_size} ({BATCH_SIZE} per GPU)")

    # Step 3: Build model and move to this rank's GPU
    model = TransformerClassifier().to(local_rank)

    # Step 4: Wrap with DDP
    # This is the only change to your model — one line.
    model = DDP(model, device_ids=[local_rank])

    # Step 5: Dataset — same data on all processes
    # (In real training, load from disk the same way on each process)
    torch.manual_seed(42)  # Same seed so all processes get the same dataset
    data = torch.randint(0, VOCAB_SIZE, (DATASET_SIZE, SEQ_LEN))
    labels = torch.randint(0, 2, (DATASET_SIZE,))
    dataset = TensorDataset(data, labels)

    # DistributedSampler: ensures each GPU sees non-overlapping samples
    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
        seed=42
    )
    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=2,
        pin_memory=True  # Faster CPU→GPU transfers
    )

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    step = 0
    session_start = time.perf_counter()
    throughputs = []

    # Step 6: Training loop — almost identical to single-GPU
    for epoch in range(NUM_EPOCHS):
        # CRITICAL: must call set_epoch so shuffle is different each epoch
        sampler.set_epoch(epoch)

        epoch_loss = 0.0
        for batch_data, batch_labels in dataloader:
            t0 = time.perf_counter()

            batch_data = batch_data.to(local_rank)
            batch_labels = batch_labels.to(local_rank)

            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()  # ← AllReduce happens HERE automatically
            optimizer.step()

            elapsed = time.perf_counter() - t0
            throughput = (BATCH_SIZE * world_size) / elapsed  # total across all GPUs
            throughputs.append(throughput)
            epoch_loss += loss.item()
            step += 1

        if rank == 0:
            avg_loss = epoch_loss / len(dataloader)
            avg_tp = sum(throughputs[-len(dataloader):]) / len(dataloader)
            mem_0 = torch.cuda.memory_allocated(0) / 1e9
            mem_1 = torch.cuda.memory_allocated(1) / 1e9
            print(f"[Epoch {epoch}] loss={avg_loss:.4f} | "
                  f"throughput={avg_tp:.0f} samples/s | "
                  f"GPU0={mem_0:.1f}GB GPU1={mem_1:.1f}GB")

    if rank == 0:
        total_time = time.perf_counter() - session_start
        avg_tp = sum(throughputs) / len(throughputs)
        print(f"\nDone. Total: {total_time:.1f}s | Avg throughput: {avg_tp:.0f} samples/s")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

## Part 1: Run DDP Training

In [ ]:
# torchrun launches one process per GPU automatically
# --nproc_per_node=2 means: 2 processes on this node (= 2 GPUs)
!torchrun --nproc_per_node=2 /kaggle/working/ddp_train.py

**What to observe:**
- Both GPUs show memory allocation
- Throughput should be ~1.8x your Module 00 baseline (not 2x — there's communication overhead)
- Loss decreases across epochs

---

## Part 2: Verify Gradient Synchronization

DDP guarantees that after each backward pass, all GPU copies have identical gradients. Let's verify this.

In [ ]:
%%writefile /kaggle/working/ddp_verify_grads.py
"""
Verifies that DDP keeps model parameters in sync across GPUs.
After training, the model on GPU 0 and GPU 1 should have identical weights.
"""
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def main():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)

    model = nn.Linear(64, 10).to(local_rank)
    model = DDP(model, device_ids=[local_rank])

    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    # Use DIFFERENT inputs on each rank (that is the whole point of DDP)
    torch.manual_seed(rank * 100)  # Different seed per rank
    x = torch.randn(16, 64).to(local_rank)
    y = torch.randint(0, 10, (16,)).to(local_rank)

    # One training step
    optimizer.zero_grad()
    loss = nn.CrossEntropyLoss()(model(x), y)
    loss.backward()  # ← AllReduce syncs grads here
    optimizer.step()

    # Extract the weight from this rank
    my_weight = model.module.weight.data.clone()  # .module unwraps DDP

    # Gather weights from all ranks onto rank 0
    all_weights = [torch.zeros_like(my_weight) for _ in range(world_size)]
    dist.all_gather(all_weights, my_weight)

    if rank == 0:
        are_equal = torch.allclose(all_weights[0], all_weights[1])
        print(f"GPU 0 and GPU 1 weights are identical: {are_equal}")
        if not are_equal:
            diff = (all_weights[0] - all_weights[1]).abs().max().item()
            print(f"Max difference: {diff}")
        print("DDP verification passed!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/ddp_verify_grads.py

## Part 3: The Bug — Missing set_epoch()

This exercise shows what goes wrong when you forget `sampler.set_epoch(epoch)`.

In [ ]:
%%writefile /kaggle/working/ddp_bug_demo.py
"""
Demonstrates the duplicate-data bug when set_epoch() is missing.
"""
import os
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

def main():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    dist.init_process_group(backend="nccl")

    dataset = TensorDataset(torch.arange(20), torch.zeros(20))
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=0)
    loader = DataLoader(dataset, batch_size=4, sampler=sampler)

    print(f"\n=== WITHOUT set_epoch() ===")
    for epoch in range(3):
        # BUG: not calling sampler.set_epoch(epoch)
        first_batch = next(iter(loader))[0].tolist()
        print(f"  [Rank {rank}] Epoch {epoch}: first batch = {first_batch}")
    # Notice: same indices every epoch on each rank!

    print(f"\n=== WITH set_epoch() ===")
    for epoch in range(3):
        sampler.set_epoch(epoch)  # FIX
        first_batch = next(iter(loader))[0].tolist()
        print(f"  [Rank {rank}] Epoch {epoch}: first batch = {first_batch}")
    # Notice: different indices each epoch

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/ddp_bug_demo.py

## Part 4: Gradient Accumulation with DDP

Gradient accumulation lets you simulate a larger batch size without using more memory.
With DDP, there is a subtlety: you should only sync gradients on the final accumulation step.

In [ ]:
%%writefile /kaggle/working/ddp_grad_accum.py
"""
Gradient accumulation with DDP.

Use model.no_sync() context manager to skip gradient AllReduce
on intermediate accumulation steps. This reduces communication overhead.
"""
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler

BATCH_SIZE = 16
ACCUM_STEPS = 4      # Effective batch = 16 * 4 * 2 GPUs = 128
VOCAB_SIZE = 10000
SEQ_LEN = 64

def main():
    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)

    model = nn.Sequential(
        nn.Embedding(VOCAB_SIZE, 256),
        nn.TransformerEncoderLayer(256, nhead=8, batch_first=True),
        nn.Linear(256, 2)
    ).to(local_rank)
    model = DDP(model, device_ids=[local_rank])
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    dataset = TensorDataset(
        torch.randint(0, VOCAB_SIZE, (400, SEQ_LEN)),
        torch.randint(0, 2, (400,))
    )
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler)

    optimizer.zero_grad()
    step = 0

    for i, (x, y) in enumerate(loader):
        x, y = x.to(local_rank), y.to(local_rank)
        is_last_accum = ((i + 1) % ACCUM_STEPS == 0)

        if is_last_accum:
            # Normal: AllReduce will fire during backward
            out = model(x)
        else:
            # Disable AllReduce for intermediate steps
            # This avoids wasting communication bandwidth
            with model.no_sync():
                out = model(x)

        out = out[:, 0, :]  # CLS token
        loss = nn.CrossEntropyLoss()(out, y) / ACCUM_STEPS
        loss.backward()

        if is_last_accum:
            optimizer.step()
            optimizer.zero_grad()
            if rank == 0:
                print(f"[Optimizer step {step}] loss={loss.item() * ACCUM_STEPS:.4f}")
            step += 1

        if step >= 5:
            break

    if rank == 0:
        print(f"\nEffective batch size: {BATCH_SIZE * ACCUM_STEPS * world_size}")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

In [ ]:
!torchrun --nproc_per_node=2 /kaggle/working/ddp_grad_accum.py

## Checkpoint Questions

1. Where exactly does gradient synchronization happen in DDP? (which line in the script?)
2. Why is throughput ~1.8x and not exactly 2x?
3. What does `model.no_sync()` do and why does it help?
4. What is the effective batch size in the gradient accumulation script?

**Answers:**
1. During `loss.backward()` — DDP hooks fire AllReduce as each gradient is computed.
2. AllReduce communication overhead and CUDA kernel launch overhead reduce the ideal 2x speedup.
3. `no_sync()` disables gradient AllReduce for intermediate accumulation steps, reducing communication by a factor of `ACCUM_STEPS`.
4. `16 * 4 * 2 = 128` samples per optimizer step.

---

## Next: Open `../02_model_parallelism/notebook.ipynb`